In [1]:
print("HEllo world")

HEllo world


## Docs, References
https://reference.langchain.com/python/langchain-core/runnables/base/Runnable

In [10]:
from helpers.common import (
  BASE_URL,
  DEEPSEEK_MODEL,
  QWEN_MODEL,
  langfuse,
  langfuse_handler,
)

In [22]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate


In [23]:
llm = ChatOllama(
  model=QWEN_MODEL,
  base_url=BASE_URL,
  validate_model_on_init=True
)

In [24]:
system_prompt = SystemMessagePromptTemplate.from_template("You are a {level} teacher. Answer in short points.")
human_prompt = HumanMessagePromptTemplate.from_template("Tell me about {topic} in {number} points.")
chat_prompt = ChatPromptTemplate([system_prompt, human_prompt])
chat_prompt

ChatPromptTemplate(input_variables=['level', 'number', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['level'], input_types={}, partial_variables={}, template='You are a {level} teacher. Answer in short points.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['number', 'topic'], input_types={}, partial_variables={}, template='Tell me about {topic} in {number} points.'), additional_kwargs={})])

## Without Chain

In [ ]:
# final_chat_prompt = chat_prompt.invoke({
#   'language': "hindi"
# })

# final_chat_prompt
# async for message in llm.astream(
#   final_chat_prompt,
#   config={"callbacks": [langfuse_handler], "run_name": "ollama-translator"},
# ):
#   print(message.content, end="")

ChatPromptValue(messages=[SystemMessage(content='You are a hindi translator. Translate the following into hindi but write in english language', additional_kwargs={}, response_metadata={}), HumanMessage(content='Good Morning People!', additional_kwargs={}, response_metadata={})])

# With Chain
### Sequential

In [25]:
chain = chat_prompt | llm
chain

ChatPromptTemplate(input_variables=['level', 'number', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['level'], input_types={}, partial_variables={}, template='You are a {level} teacher. Answer in short points.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['number', 'topic'], input_types={}, partial_variables={}, template='Tell me about {topic} in {number} points.'), additional_kwargs={})])
| ChatOllama(model='qwen2.5:7b', validate_model_on_init=True, base_url='http://localhost:11434/')

In [27]:
for message in chain.stream({
  "level": "school",
  "topic": "Earth",
  "number": "2",

},
  config={"callbacks": [langfuse_handler], "run_name": "school-teacher"},
):
  print(message.content, end="")


1. Earth is the third planet from the Sun and is the only known celestial body to support life.
2. It has diverse ecosystems, including forests, oceans, deserts, and polar regions, supporting a wide variety of plant and animal species.

# Chain Sequential | StrOutputParser

In [30]:
from langchain_core.output_parsers import StrOutputParser
chain_2 = chat_prompt | llm | StrOutputParser()
chain_2

ChatPromptTemplate(input_variables=['level', 'number', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['level'], input_types={}, partial_variables={}, template='You are a {level} teacher. Answer in short points.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['number', 'topic'], input_types={}, partial_variables={}, template='Tell me about {topic} in {number} points.'), additional_kwargs={})])
| ChatOllama(model='qwen2.5:7b', validate_model_on_init=True, base_url='http://localhost:11434/')
| StrOutputParser()

In [32]:
for message in chain_2.stream({
  "level": "school",
  "topic": "Earth",
  "number": "2",

},
  config={"callbacks": [langfuse_handler], "run_name": "school-teacher"},
):
  print(message, end="")


1. Earth is the third planet from the Sun and the only astronomical object known to support life.
2. It is primarily composed of rock and water, with a diverse range of ecosystems and habitats supporting millions of species.

In [28]:
langfuse.flush()